# Model Evaluation Metrics

Accuracy is misleading. For imbalanced data or different costs, you need the right metric.

## Confusion Matrix

Foundation for all classification metrics. Always plot this first — accuracy hides problems.

- **True Positive (TP)**: Correctly predicted positive
- **False Positive (FP)**: Incorrectly predicted positive  
- **True Negative (TN)**: Correctly predicted negative
- **False Negative (FN)**: Incorrectly predicted negative

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
# Create imbalanced dataset
X, y = make_classification(n_samples=1000, n_features=2, n_classes=2, 
                          n_redundant=0, n_clusters_per_class=1, 
                          weights=[0.9, 0.1], random_state=42)  # 90% class 0, 10% class 1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of positive class

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives (TN): {cm[0,0]}")
print(f"False Positives (FP): {cm[0,1]}")
print(f"False Negatives (FN): {cm[1,0]}")
print(f"True Positives (TP): {cm[1,1]}")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## Precision vs Recall

- **Precision**: TP/(TP+FP) — Of predicted positives, how many correct?
- **Recall**: TP/(TP+FN) — Of actual positives, how many found?
- **F1**: Harmonic mean of precision and recall

Choose based on business cost.

In [ ]:
# Calculate metrics manually
TN, FP, FN, TP = cm.ravel()

accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

# Demonstrate precision-recall tradeoff
thresholds = np.linspace(0, 1, 100)
precisions = []
recalls = []

for threshold in thresholds:
    y_pred_thresh = (y_prob >= threshold).astype(int)
    cm_thresh = confusion_matrix(y_test, y_pred_thresh)
    
    if cm_thresh.shape == (2, 2):
        tn, fp, fn, tp = cm_thresh.ravel()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        precisions.append(prec)
        recalls.append(rec)
    else:
        precisions.append(0)
        recalls.append(0)

# Plot precision-recall curve
plt.figure(figsize=(10, 6))
plt.plot(recalls, precisions, 'b-', linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True, alpha=0.3)
plt.show()

## ROC & AUC

ROC plots TPR vs FPR at different thresholds. AUC measures discriminative ability.

Good for balanced classes. Use precision-recall for imbalanced data.

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 6))

# ROC curve
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linewidth=2, linestyle='--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)

# Threshold vs metrics
plt.subplot(1, 2, 2)
plt.plot(thresholds, precisions[:len(thresholds)], 'b-', label='Precision', linewidth=2)
plt.plot(thresholds, recalls[:len(thresholds)], 'r-', label='Recall', linewidth=2)
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"AUC Score: {roc_auc:.3f}")
print("\nAUC Interpretation:")
print("- 0.5: Random guessing")
print("- 0.7-0.8: Acceptable")
print("- 0.8-0.9: Good")
print("- 0.9-1.0: Excellent")

## Regression Metrics

- **MAE**: Mean absolute error (robust to outliers)
- **MSE**: Mean squared error (penalises large errors)
- **RMSE**: Root mean squared error (interpretable units)
- **R²**: Explained variance (0–1, higher better)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Generate regression data
X_reg, y_reg = make_classification(n_samples=200, n_features=1, n_classes=1, 
                                  n_redundant=0, n_clusters_per_class=1, random_state=42)
y_reg = y_reg + np.random.normal(0, 0.5, len(y_reg))  # Add noise

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

# Train model
reg_model = LinearRegression()
reg_model.fit(X_train_reg, y_train_reg)

# Predictions
y_pred_reg = reg_model.predict(X_test_reg)

# Calculate metrics
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"Mean Absolute Error (MAE): {mae:.3f}")
print(f"Mean Squared Error (MSE): {mse:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
print(f"R² Score: {r2:.3f}")

# Visualize predictions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test_reg, y_pred_reg, alpha=0.7)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 
         'r--', linewidth=2, label='Perfect predictions')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Predictions vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)

# Residuals plot
residuals = y_test_reg - y_pred_reg
plt.subplot(1, 2, 2)
plt.scatter(y_pred_reg, residuals, alpha=0.7)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRegression Metrics Interpretation:")
print("- MAE: Average absolute prediction error")
print("- MSE: Penalises large errors more (squared)")
print("- RMSE: In same units as target variable")
print("- R²: Fraction of variance explained (1.0 = perfect)")

## Key Takeaways

- **Accuracy fails on imbalanced data**. Use confusion matrix always.
- **Precision** important when false positives costly (spam filter).
- **Recall** important when false negatives costly (medical diagnosis).
- **F1** balances precision and recall. Use when both matter equally.
- **AUC** good for ranking. Use when you care about prediction quality across thresholds.
- **Cross-validation** gives reliable estimates. Never evaluate on training data.

## Exercises

1. **Medical Diagnosis**: You build a cancer detection model. Which metric matters most: precision or recall? Why?

2. **Spam Filter**: For an email spam classifier, which is worse: false positive or false negative? Which metric should you optimize?

3. **AUC vs Accuracy**: When would you prefer AUC over accuracy as your evaluation metric?

4. **Regression Evaluation**: Your house price model has R² = 0.85. Is this good? What does it mean?

5. **Threshold Tuning**: You have a fraud detection model with 80% recall but only 20% precision. How can you improve precision without retraining?

## Solutions

1. **Medical Diagnosis**: **Recall** matters most. False negatives (missing cancer) are catastrophic. Better to have false positives (unnecessary tests) than miss actual cases.

2. **Spam Filter**: False positive (legit email marked as spam) is worse — users miss important emails. Optimize for **precision** to minimize false positives.

3. **AUC vs Accuracy**: Use AUC for imbalanced datasets, when you care about ranking quality, or when threshold will be tuned later. Accuracy fails when classes are imbalanced.

4. **Regression Evaluation**: R² = 0.85 is quite good (explains 85% of variance). Excellent > 0.9, Good > 0.8, Fair > 0.7, Poor < 0.7.

5. **Threshold Tuning**: Increase the classification threshold. Currently at 0.5, raising it to 0.8 will reduce false positives (increase precision) but may also reduce true positives (decrease recall).